<a href="https://colab.research.google.com/github/maierav/ai_oscp_neuro/blob/main/notebooks/robustness_diagnostics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Robustness diagnostics — animal-level support for every primary effect

Three complementary sampling-honesty checks for the prediction-error results, all
computed from committed per-unit tables in `data/` (no DANDI streaming needed):

1. **Area × layer prediction error and its animal support** — median DvI per
   area×layer cell *and the number of contributing mice*; cells with <3 mice are
   hatched. Shows that outside VISp most anatomical cells rest on 1–5 animals.
2. **Per-animal forest plot** — each mouse's median as a point, the pooled
   hierarchical-bootstrap CI (◆), the leave-one-animal-out range (⊢), and an
   **exact sign test on per-animal medians** (the most conservative honest test,
   N = mice) for each primary effect.
3. **Result 3 behavioral balance** — closed- vs open-loop running events matched
   on running speed and pupil arousal but severely imbalanced on time-in-session
   (the block-order confound, quantified).

Inputs (`data/`): `oddball_confirmatory_units.parquet`, `sequence_units.parquet`,
`duration_units_layered.parquet`, `sensorimotor_multisession_units.parquet`,
`sensorimotor_behavioral_balance.parquet`. The balance cell can re-stream the raw
covariates from DANDI with `REBUILD=True`.

In [ ]:
# In Colab, uncomment to fetch the repo data:
# !pip -q install pynwb remfile h5py
# !git clone -q https://github.com/maierav/ai_oscp_neuro.git && cd ai_oscp_neuro/notebooks
import os, numpy as np, pandas as pd, matplotlib.pyplot as plt, matplotlib as mpl
from matplotlib.patches import Rectangle
from scipy import stats as ss

# repo data dir (works from notebooks/ locally and from the repo root in Colab)
DATA = "../data" if os.path.isdir("../data") else "data"
LAYERS = ["L2/3", "L4", "L5", "L6"]
AREAS_ALL = ["VISp", "VISlm", "VISl", "VISa", "VISrl"]

def lbin(l):
    l = str(l)
    if l in ("1", "2", "2/3", "3"): return "L2/3"
    if l == "4": return "L4"
    if l == "5": return "L5"
    if l.startswith("6"): return "L6"
    return None

import re
def areaf(a):
    m = re.match(r"(VIS[a-z]+|VIS[A-Z]*)", str(a)) or re.match(r"([A-Za-z]+)", str(a))
    return m.group(1) if m else str(a)

# ---- per-unit prediction-error index per paradigm ----
OD = pd.read_parquet(f"{DATA}/oddball_confirmatory_units.parquet")
OD["idx"] = OD["DvI_90"]
SEQ = pd.read_parquet(f"{DATA}/sequence_units.parquet")
SEQ["idx"] = (SEQ.R_odd90 - SEQ.R_c90) / (SEQ.R_odd90.abs() + SEQ.R_c90.abs() + 1e-9)
DUR = pd.read_parquet(f"{DATA}/duration_units_layered.parquet")
DUR["idx"] = DUR.om_pe / (DUR.om_pe.abs() + DUR.std_r.abs() + 1e-9)
SMU = pd.read_parquet(f"{DATA}/sensorimotor_multisession_units.parquet")
_sm = SMU.dropna(subset=["motor_orientation_90_cl", "motor_orientation_90_ol"]).copy()
_c, _o = _sm.motor_orientation_90_cl, _sm.motor_orientation_90_ol
_sm["idx"] = (_c - _o) / (_c.abs() + _o.abs() + 1e-9)
SM = _sm[["subject", "idx"]].copy()
for df in (OD, SEQ, DUR, SM):
    df["subject"] = df["subject"].astype(str)
print("loaded per-unit tables:",
      {"oddball": len(OD), "sequence": len(SEQ), "duration": len(DUR), "sensorimotor": len(SM)})

## 1. Area × layer prediction error with contributing-animal counts

In [ ]:
def agg_area_layer(df):
    d = df.copy()
    d["A"] = d["area"].map(areaf); d["L"] = d["layer"].map(lbin)
    d = d[d.A.str.startswith("VIS") & d.L.notna()]
    return d.groupby(["A", "L"]).agg(dvi=("idx", "median"),
                                     n_units=("idx", "size"),
                                     n_animals=("subject", lambda s: s.nunique())).reset_index()

AGG = {"Feature-oddball": agg_area_layer(OD),
       "Sequence": agg_area_layer(SEQ),
       "Duration / timing": agg_area_layer(DUR)}

def _grid(g):
    areas = [a for a in AREAS_ALL if a in set(g.A)]
    D = np.full((len(LAYERS), len(areas)), np.nan)
    N = np.zeros((len(LAYERS), len(areas)), int); A = np.zeros_like(N)
    for _, r in g.iterrows():
        if r.A in areas:
            i = LAYERS.index(r.L); j = areas.index(r.A)
            D[i, j] = r.dvi; N[i, j] = r.n_units; A[i, j] = r.n_animals
    return areas, D, N, A

fig, axes = plt.subplots(1, 3, figsize=(12.5, 3.9))
fig.subplots_adjust(left=0.055, right=0.90, top=0.80, bottom=0.16, wspace=0.30)
cmap = plt.cm.RdBu_r; norm = mpl.colors.TwoSlopeNorm(vmin=-0.6, vcenter=0.0, vmax=0.9); im = None
for ax, (title, g) in zip(axes, AGG.items()):
    areas, D, N, A = _grid(g)
    im = ax.imshow(D, cmap=cmap, norm=norm, aspect="auto")
    ax.set_xticks(range(len(areas))); ax.set_xticklabels(areas, fontsize=7)
    ax.set_yticks(range(len(LAYERS))); ax.set_yticklabels(LAYERS if ax is axes[0] else [""] * len(LAYERS), fontsize=7)
    ax.set_title(title, loc="left", fontsize=9)
    for i in range(len(LAYERS)):
        for j in range(len(areas)):
            if np.isnan(D[i, j]):
                ax.add_patch(Rectangle((j - 0.5, i - 0.5), 1, 1, facecolor="#eeeeee", edgecolor="white")); continue
            na = int(A[i, j])
            if na < 3:
                ax.add_patch(Rectangle((j - 0.5, i - 0.5), 1, 1, fill=False, hatch="////", edgecolor="#555555", lw=0))
            tc = "white" if abs(norm(D[i, j]) - 0.5) > 0.32 else "black"
            ax.text(j, i - 0.13, f"{D[i, j]:+.2f}", ha="center", va="center", fontsize=7, color=tc, fontweight="bold")
            ax.text(j, i + 0.24, f"{na} {'mouse' if na == 1 else 'mice'}", ha="center", va="center", fontsize=5.8, color=tc)
    ax.set_xlim(-0.5, len(areas) - 0.5); ax.set_ylim(len(LAYERS) - 0.5, -0.5)
    for s in ax.spines.values(): s.set_visible(False)
axes[0].set_ylabel("cortical layer", fontsize=8)
cax = fig.add_axes([0.915, 0.16, 0.016, 0.64]); cb = fig.colorbar(im, cax=cax)
cb.set_label("prediction-error index (median DvI)", fontsize=7.5); cb.ax.tick_params(labelsize=6.5)
fig.suptitle("Area × layer prediction error and its animal support (hatched = <3 mice — under-sampled, do not over-read)",
             fontsize=10, x=0.055, ha="left", y=0.95)
plt.show()

## 2. Per-animal forest — leave-one-out + exact animal-level tests

In [ ]:
EFFECTS = {
    "Feature-oddball\n(DvI\u2089\u2080)": OD[["subject", "idx"]].copy(),
    "Sequence\n(DvI\u2089\u2080)": SEQ[["subject", "idx"]].copy(),
    "Duration / timing\n(timing-PE)": DUR[["subject", "idx"]].copy(),
    "Sensorimotor\n(closed\u2212open DvI\u2089\u2080, null)": SM,
}

def boot_hier(df, n=5000, seed=42):
    rng = np.random.default_rng(seed); subs = df.subject.unique()
    by = {s: df[df.subject == s]["idx"].values for s in subs}
    est = [np.median(np.concatenate([rng.choice(by[s], len(by[s]), replace=True)
                                      for s in rng.choice(subs, len(subs), replace=True)])) for _ in range(n)]
    return np.median(df["idx"]), np.percentile(est, 2.5), np.percentile(est, 97.5)

SUMMARY = {}
for name, df in EFFECTS.items():
    g = df.groupby("subject")["idx"]; pa_med, pa_n = g.median(), g.size()
    pooled, lo, hi = boot_hier(df)
    loo = np.array([np.median(df[df.subject != s]["idx"]) for s in pa_med.index])
    npos, N = int((pa_med > 0).sum()), len(pa_med)
    sign_p = ss.binomtest(npos, N, 0.5).pvalue
    try: w_p = ss.wilcoxon(pa_med.values).pvalue
    except Exception: w_p = np.nan
    SUMMARY[name] = dict(pa_med=pa_med, pa_n=pa_n, pooled=pooled, lo=lo, hi=hi,
                         loo_min=loo.min(), loo_max=loo.max(), npos=npos, N=N, sign_p=sign_p, w_p=w_p)

colors = {"Feature-oddball\n(DvI\u2089\u2080)": "#b3402a", "Sequence\n(DvI\u2089\u2080)": "#2f6b46",
          "Duration / timing\n(timing-PE)": "#2b5d8a", "Sensorimotor\n(closed\u2212open DvI\u2089\u2080, null)": "#777777"}
fig, ax = plt.subplots(figsize=(9.2, 6.0)); fig.subplots_adjust(left=0.30, right=0.74, top=0.90, bottom=0.10)
ax.axvline(0, color="#999", lw=1, ls="--", zorder=0)
ytick = []; ylab = []; y = 0; gap = 1.1
for name in EFFECTS:
    S = SUMMARY[name]; col = colors[name]; pa = S["pa_med"].sort_values(); yb0 = y - 0.4
    for s, v in pa.items():
        ms = 6 + 3 * np.sqrt(S["pa_n"][s] / 50)
        ax.scatter(v, y, s=ms ** 1.4, color=col, alpha=0.8, zorder=3, edgecolors="white", linewidths=0.5); y += 0.32
    ygrp = (yb0 + y - 0.32 + 0.4) / 2; ysum = y + 0.12
    ax.plot([S["lo"], S["hi"]], [ysum, ysum], color=col, lw=3, zorder=4, solid_capstyle="round")
    ax.scatter([S["pooled"]], [ysum], marker="D", s=55, color=col, zorder=5, edgecolors="black", linewidths=0.6)
    ax.plot([S["loo_min"], S["loo_max"]], [ysum - 0.26, ysum - 0.26], color=col, lw=1.2, alpha=0.7, zorder=4)
    for xx in (S["loo_min"], S["loo_max"]):
        ax.plot([xx, xx], [ysum - 0.32, ysum - 0.20], color=col, lw=1.2, alpha=0.7)
    ytick.append(ygrp); ylab.append(name)
    star = "\u2713" if S["sign_p"] < 0.05 else "\u25cb"
    ax.text(0.755, ysum, f"{S['npos']}/{S['N']} mice +   sign p={S['sign_p']:.3f}  {star}",
            transform=ax.get_yaxis_transform(), va="center", ha="left", fontsize=7.2, color=col)
    y = ysum + gap
ax.set_yticks(ytick); ax.set_yticklabels(ylab, fontsize=8)
ax.set_ylim(-0.6, y - gap + 0.5); ax.invert_yaxis()
ax.set_xlabel("prediction-error index  (per-animal median \u25cf · pooled hierarchical CI \u25c6 · leave-one-out range \u22a2)", fontsize=8)
ax.set_xlim(-0.35, 0.95)
for sp in ["top", "right", "left"]: ax.spines[sp].set_visible(False)
ax.tick_params(left=False)
fig.suptitle("Per-animal robustness of every primary effect", fontsize=10, x=0.30, ha="left", y=0.965)
plt.show()
print(pd.DataFrame([{"effect": k.splitlines()[0], "n_animals": v["N"], "n_pos": v["npos"],
                     "sign_p": round(v["sign_p"], 3)} for k, v in SUMMARY.items()]).to_string(index=False))

## 3. Result 3 behavioral balance (speed / pupil / time-in-session / block order)

In [ ]:
# REBUILD=False (default): re-plot from the committed per-event covariate table (fast, offline).
# REBUILD=True: re-stream the 6 sessions from DANDI and recompute speed/pupil/time per event.
REBUILD = False
BAL = f"{DATA}/sensorimotor_behavioral_balance.parquet"
if REBUILD:
    import openscope_ccf as o, h5py, remfile
    PRE = [("848387", "2026-05-04-16-19-01"), ("830848", "2026-03-05-16-07-50"),
           ("830794", "2026-01-26-12-02-05"), ("832691", "2026-03-26-10-51-37"),
           ("830847", "2026-03-12-17-14-42"), ("834686", "2026-03-26-16-15-13")]
    DEVS = ["motor_orientation_90", "motor_orientation_45", "motor_halt", "motor_omission"]
    def _col(g, n):
        a = g[n][:]; return np.array([x.decode() if isinstance(x, bytes) else x for x in a])
    rows = []
    for subj, date in PRE:
        fh = h5py.File(remfile.File(o.s3_url(o.resolve_asset(subj, date))), "r")
        g = fh["intervals"]["Sensory-motor mismatch block_presentations"]
        TT = _col(g, "TrialType"); ts = g["start_time"][:]
        ol = next(fh["intervals"][k] for k in fh["intervals"].keys()
                  if "BlockType" in fh["intervals"][k] and "open_loop_prerecorded" in set(_col(fh["intervals"][k], "BlockType")))
        oTT = _col(ol, "TrialType"); ots = ol["start_time"][:]
        rs = fh["processing"]["running"]["running_speed"]; rt = rs["timestamps"][:]; rv = rs["data"][:]
        pt = pv = None
        try:
            pg = fh["processing"]["eye_tracking"]["pupil"]; pt = pg["timestamps"][:]; pv = pg["area"][:]
        except Exception: pass
        t0, t1 = float(ts.min()), float(max(ts.max(), ots.max()))
        def cov(t):
            m = (rt >= t - 1) & (rt < t); spd = float(np.abs(rv[m]).mean()) if m.any() else np.nan
            pup = np.nan
            if pt is not None:
                pm = (pt >= t - 1) & (pt < t); vals = pv[pm][np.isfinite(pv[pm])] if pm.any() else []
                pup = float(np.nanmedian(vals)) if len(vals) else np.nan
            return spd, pup, (t - t0) / (t1 - t0)
        for arm, (A, T) in [("closed", (TT, ts)), ("open", (oTT, ots))]:
            for d in DEVS:
                for t in T[A == d]:
                    spd, pup, frac = cov(float(t))
                    if not np.isnan(spd) and spd > 1.0:
                        rows.append(dict(subject=subj, arm=arm, deviant=d, t=float(t), speed=spd, pupil=pup, frac_session=frac))
        fh.close()
    B = pd.DataFrame(rows); B.to_parquet(BAL, index=False)
else:
    B = pd.read_parquet(BAL)

CC = {"closed": "#c0392b", "open": "#3b6ea5"}
fig, axes = plt.subplots(1, 4, figsize=(13, 3.7)); fig.subplots_adjust(left=0.055, right=0.985, top=0.80, bottom=0.17, wspace=0.42)
def _bp(ax, var, label, unit):
    c = B[B.arm == "closed"][var].dropna(); ov = B[B.arm == "open"][var].dropna()
    for i, (arm, vals) in enumerate([("closed", c), ("open", ov)]):
        x = np.random.default_rng(i).normal(i, 0.06, len(vals))
        ax.scatter(x, vals, s=5, color=CC[arm], alpha=0.35, linewidths=0, zorder=1)
        ax.plot([i - 0.22, i + 0.22], [np.median(vals)] * 2, color="black", lw=2, zorder=3)
    p = ss.mannwhitneyu(c, ov).pvalue
    ax.set_xticks([0, 1]); ax.set_xticklabels(["closed\nrun", "open\nrun"], fontsize=7)
    ax.set_ylabel(f"{label} ({unit})", fontsize=8)
    ax.set_title(f"{label}\nMW p={p:.2g}" + ("  \u2713 matched" if p > 0.05 else "  \u2717 IMBALANCED"),
                 loc="left", fontsize=8.5, color=("#2c7a2c" if p > 0.05 else "#b03030"))
    ax.margins(y=0.06)
_bp(axes[0], "speed", "running speed", "cm/s")
_bp(axes[1], "pupil", "pupil area", "px\u00b2")
_bp(axes[2], "frac_session", "time in session", "fraction 0\u20131")
axD = axes[3]
for subj in sorted(B.subject.unique()):
    b = B[B.subject == subj]
    mc = b[b.arm == "closed"].frac_session.median(); mo = b[b.arm == "open"].frac_session.median()
    axD.plot([0, 1], [mc, mo], "-", color="#888", lw=0.8, zorder=1)
    axD.scatter([0], [mc], s=22, color=CC["closed"], zorder=2); axD.scatter([1], [mo], s=22, color=CC["open"], zorder=2)
axD.set_xticks([0, 1]); axD.set_xticklabels(["closed\nrun", "open\nrun"], fontsize=7)
axD.set_ylabel("median time in session (fraction)", fontsize=8); axD.set_ylim(0, 1.02)
axD.set_title("per-session block order\n(open-loop always late)", loc="left", fontsize=8.5, color="#b03030")
axD.margins(y=0.06)
fig.suptitle("Result 3 behavioral balance — closed- vs open-loop running events: speed & arousal matched, block order is not",
             fontsize=9.5, x=0.055, ha="left", y=0.955)
plt.show()

## Takeaway

- **Feature-oddball** (9/9 mice +, sign p≈0.004) and **duration/timing** (6/6, p≈0.031) survive the exact animal-level test and barely move under leave-one-out — robust.
- **Sequence** (6/7, p≈0.13) has a pooled CI excluding zero but does *not* reach the conservative animal-level test at n=7 mice.
- **Sensorimotor** (5/6, p≈0.22, closed−open index) is consistent with the Result 3 null.
- The heatmap shows only VISp is broadly animal-supported; the behavioral panel shows the closed/open contrast is matched on state but confounded by block order.